# Extract Skills from Job Descriptions (Multi-File Support - OPTIMIZED)

This notebook processes multiple parquet files with **high-performance parallel processing**.

**Performance Optimizations:**
- 🚀 Multi-core CPU parallelization (auto-detects optimal worker count)
- 🎯 GPU batch processing (optimized for A100)
- ⚡ Expected speedup: **100-500x** vs serial processing
- 💾 Low memory usage (processes one file at a time)

**Features:**
- Handles hundreds of small parquet files
- Processes files one by one (low memory usage)
- Incremental saving (results saved after each file)
- Resume support (skip already processed files)
- Progress tracking with performance metrics

**Hardware Utilization:**
- **CPU**: Uses all cores efficiently (default: 12 workers for 24-core system)
- **GPU**: Automatically uses CUDA if available (A100-80GB detected)
- **Expected throughput**: 10-50 JDs/sec (vs 0.1 JDs/sec serial)

**Use Case:** Extract skills from 20M+ job descriptions split across multiple files.

## Setup

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from glob import glob
from tqdm import tqdm
import json
import os
from skillner.jd_skill_extractor_optimized import OptimizedJobDescriptionSkillExtractor

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

print("✓ Imports successful")

## Configuration

Edit these paths to match your data:

In [2]:
# =====================================================================
# EDIT THESE PATHS
# =====================================================================

# Input folder with multiple parquet files
INPUT_FOLDER = '../JD'  # Folder containing 200-300 small parquet files

# Knowledge base (created from ONET)
KB_PATH = '../.skillner-kb/MERGED_EN.pkl'  # Or ONET_EN.pkl

# Output
OUTPUT_PATH = '../data/jd_extracted_skills.parquet'
CHECKPOINT_FILE = '../data/processing_checkpoint.json'  # Track processed files

# Column names in your data
JD_TEXT_COLUMN = 'job_description'  # Column with job description text
ONET_CODE_COLUMN = 'onet_code'
DATE_COLUMN = 'post_date'

# Extraction parameters
SIMILARITY_THRESHOLD = 0.6  # Lower = more matches, higher = stricter
MAX_WINDOW_SIZE = 5  # Maximum words in a skill phrase

# Performance optimization (NEW)
BATCH_SIZE = 128  # GPU batch size (128 optimal for A100, 64 for smaller GPUs)
NUM_WORKERS = None  # CPU workers (None = auto-detect, 12 for 24-core system)
USE_MULTIPROCESSING = True  # Enable parallel processing

# Processing options
SAVE_EVERY_N_FILES = 1  # Save results after every N files (1 = after each file)
RESUME_FROM_CHECKPOINT = True  # Skip already processed files

# =====================================================================

print(f"Configuration:")
print(f"  Input folder: {INPUT_FOLDER}")
print(f"  KB: {KB_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Checkpoint: {CHECKPOINT_FILE}")
print(f"  Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"  Max window size: {MAX_WINDOW_SIZE}")
print(f"  GPU batch size: {BATCH_SIZE}")
print(f"  CPU workers: {NUM_WORKERS or 'auto-detect'}")
print(f"  Use multiprocessing: {USE_MULTIPROCESSING}")
print(f"  Resume from checkpoint: {RESUME_FROM_CHECKPOINT}")

## Step 1: Load Data

In [3]:
# Load your sampled job descriptions
print("Loading data...")
df = pd.read_parquet(INPUT_DATA)

print(f"✓ Loaded {len(df):,} job descriptions")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample:")
df.head(3)

Loading data...


FileNotFoundError: [Errno 2] No such file or directory: '../data/jd_sampled.parquet'

## Step 2: Initialize Skill Extractor

This loads the knowledge base and semantic model (may take 1-2 minutes first time):

In [5]:
# Load checkpoint if exists
processed_files = set()
if RESUME_FROM_CHECKPOINT and os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint_data = json.load(f)
        processed_files = set(checkpoint_data.get('processed_files', []))
    print(f"✓ Loaded checkpoint: {len(processed_files)} files already processed")
else:
    print("Starting from scratch (no checkpoint found)")

# Filter out already processed files
files_to_process = [f for f in input_files if f not in processed_files]

print(f"\nFiles to process: {len(files_to_process)} / {len(input_files)}")
if len(processed_files) > 0:
    print(f"Skipping {len(processed_files)} already processed files")

Loading knowledge base from ../.skillner-kb/MERGED_EN.pkl...
✓ Loaded 111,173 skills

Loading semantic model: all-MiniLM-L6-v2...


/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading semantic model: all-MiniLM-L6-v2...
Computing skill embeddings...
✓ Ready with 111173 skills
✓ Extractor ready


✓ Extractor ready!


## Step 3: Sample Check - Load One File

In [ ]:
# Load first file to check format
if len(files_to_process) > 0:
    sample_file = files_to_process[0]
    print(f"Loading sample file: {os.path.basename(sample_file)}")
    
    df_sample = pd.read_parquet(sample_file)
    
    print(f"\n✓ Loaded {len(df_sample):,} rows")
    print(f"\nColumns: {list(df_sample.columns)}")
    print(f"\nColumn check:")
    print(f"  {JD_TEXT_COLUMN}: {'✓ Found' if JD_TEXT_COLUMN in df_sample.columns else '✗ NOT FOUND'}")
    print(f"  {ONET_CODE_COLUMN}: {'✓ Found' if ONET_CODE_COLUMN in df_sample.columns else '✗ NOT FOUND'}")
    print(f"  {DATE_COLUMN}: {'✓ Found' if DATE_COLUMN in df_sample.columns else '✗ NOT FOUND'}")
    
    print(f"\nSample data:")
    display(df_sample.head(3))
    
    # Check for null values
    null_count = df_sample[JD_TEXT_COLUMN].isna().sum()
    print(f"\nNull job descriptions: {null_count} / {len(df_sample)} ({null_count/len(df_sample)*100:.1f}%)")
else:
    print("No files to process!")

## Step 4: Initialize Optimized Skill Extractor

This loads the knowledge base and semantic model (may take 1-2 minutes).

**Optimizations:**
- Auto-detects GPU (uses A100-80GB if available)
- Auto-detects optimal CPU worker count (12 for 24-core system)
- Configures GPU batch size for maximum throughput

In [ ]:
print("Initializing optimized extractor...")
print("(This may take 1-2 minutes to load the semantic model)\n")

extractor = OptimizedJobDescriptionSkillExtractor(
    kb_path=KB_PATH,
    model_name='all-MiniLM-L6-v2',
    similarity_threshold=SIMILARITY_THRESHOLD,
    max_window_size=MAX_WINDOW_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

print("\n✓ Extractor ready!")

## Step 3: Test on Single Example

Let's test on one job description first:

In [ ]:
# Get first non-null job description
test_jd = df[df[JD_TEXT_COLUMN].notna()][JD_TEXT_COLUMN].iloc[0]

print("Test Job Description:")
print("=" * 70)
print(test_jd[:500] + "..." if len(test_jd) > 500 else test_jd)
print("=" * 70)

# Extract skills
result = extractor.extract_skills(test_jd, return_details=True)

print(f"\n✓ Found {result['num_skills']} unique skills")
print(f"\nSkills by category:")
for section, skills in result['by_section'].items():
    print(f"  [{section}]: {len(skills)} skills")
    for skill in skills[:3]:
        print(f"    - {skill}")
    if len(skills) > 3:
        print(f"    ... and {len(skills) - 3} more")

print(f"\nAll extracted skills:")
for i, skill in enumerate(sorted(result['skills']), 1):
    print(f"{i:2d}. {skill}")

## Step 6: Process All Files (OPTIMIZED)

**This is the main processing loop with parallel optimization.**

Processing strategy:
- Load one file at a time (low memory usage)
- Extract skills using multi-core CPU parallelization + GPU batch processing
- Append results to output file
- Update checkpoint
- Repeat for next file

**Performance Estimates:**
- **With optimization**: ~10-50 JDs/sec (100-500x faster)
- **100K JDs**: ~30 minutes - 3 hours (vs ~5-14 hours serial)
- **1M JDs**: ~5-30 hours (vs days serial)
- **Real-time metrics**: See throughput for each file

**Hardware Usage:**
- **CPU**: All cores utilized (12 workers for 24-core system)
- **GPU**: A100-80GB batch processing (batch size 128)
- Progress is saved continuously, so you can stop and resume anytime

In [ ]:
import time

print("="*70)
print("Starting Multi-File Processing (OPTIMIZED)")
print("="*70)
print(f"Files to process: {len(files_to_process)}")
print(f"Output: {OUTPUT_PATH}")
print(f"Performance: {extractor.num_workers} CPU workers + GPU batch size {extractor.batch_size}")
print(f"You can stop this cell anytime and resume later.\n")

# Track statistics
total_jds_processed = 0
total_files_processed = 0
total_time = 0

# Process each file
for file_idx, file_path in enumerate(files_to_process, 1):
    file_name = os.path.basename(file_path)
    
    try:
        # Load file
        df_chunk = pd.read_parquet(file_path)
        
        # Skip if no job descriptions
        if JD_TEXT_COLUMN not in df_chunk.columns:
            print(f"\n⚠️ Skipping {file_name}: Column '{JD_TEXT_COLUMN}' not found")
            continue
        
        # Extract skills with optimized method
        jd_list = df_chunk[JD_TEXT_COLUMN].tolist()
        
        print(f"\nFile {file_idx}/{len(files_to_process)}: {file_name} ({len(jd_list):,} JDs)")
        
        file_start = time.time()
        results = extractor.extract_skills_batch_optimized(
            jd_list, 
            show_progress=True,
            use_multiprocessing=USE_MULTIPROCESSING
        )
        file_time = time.time() - file_start
        
        # Performance stats
        throughput = len(jd_list) / file_time if file_time > 0 else 0
        print(f"  ✓ Completed in {file_time:.1f}s ({throughput:.2f} JDs/sec)")
        
        # Convert results to DataFrame
        results_df = pd.DataFrame([
            {
                'skills': r['skills'],
                'num_skills': r['num_skills'],
                'by_section': r['by_section']
            }
            for r in results
        ])
        
        # Combine with original data
        df_combined = pd.concat([df_chunk.reset_index(drop=True), results_df], axis=1)
        
        # Add quarter column if date available
        if DATE_COLUMN in df_combined.columns:
            df_combined[DATE_COLUMN] = pd.to_datetime(df_combined[DATE_COLUMN], errors='coerce')
            df_combined['quarter'] = df_combined[DATE_COLUMN].dt.to_period('Q')
        
        # Save results (append mode)
        if file_idx == 1 and not os.path.exists(OUTPUT_PATH):
            # First file: create new file
            df_combined.to_parquet(OUTPUT_PATH, index=False)
        else:
            # Subsequent files: append
            existing_df = pd.read_parquet(OUTPUT_PATH)
            combined_output = pd.concat([existing_df, df_combined], ignore_index=True)
            combined_output.to_parquet(OUTPUT_PATH, index=False)
        
        # Update checkpoint
        processed_files.add(file_path)
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({
                'processed_files': list(processed_files),
                'total_files': len(input_files),
                'files_remaining': len(input_files) - len(processed_files)
            }, f, indent=2)
        
        # Update statistics
        total_jds_processed += len(df_chunk)
        total_files_processed += 1
        total_time += file_time
        
        # Print cumulative stats
        avg_throughput = total_jds_processed / total_time if total_time > 0 else 0
        print(f"  Cumulative: {total_jds_processed:,} JDs in {total_time:.1f}s ({avg_throughput:.2f} JDs/sec)")
        
    except Exception as e:
        print(f"\n✗ Error processing {file_name}: {e}")
        import traceback
        traceback.print_exc()
        continue

print("\n" + "="*70)
print("PROCESSING COMPLETE")
print("="*70)
print(f"Files processed: {total_files_processed}")
print(f"Job descriptions processed: {total_jds_processed:,}")
print(f"Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"Average throughput: {total_jds_processed/total_time:.2f} JDs/sec")
print(f"Output saved to: {OUTPUT_PATH}")
print(f"Checkpoint saved to: {CHECKPOINT_FILE}")

## Step 5: Combine Results with Original Data

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame([
    {
        'skills': r['skills'],
        'num_skills': r['num_skills'],
        'by_section': r['by_section']
    }
    for r in results
])

# Combine with original data
df_combined = pd.concat([df.reset_index(drop=True), results_df], axis=1)

# Create quarter column for time series analysis
if DATE_COLUMN in df_combined.columns:
    df_combined[DATE_COLUMN] = pd.to_datetime(df_combined[DATE_COLUMN])
    df_combined['quarter'] = df_combined[DATE_COLUMN].dt.to_period('Q')

print(f"Combined DataFrame shape: {df_combined.shape}")
print(f"\nSample:")
df_combined[['onet_code', 'quarter', 'num_skills']].head()

## Step 6: Basic Statistics

In [ ]:
# Get statistics
stats = extractor.get_statistics(results)

print("="*70)
print("Extraction Statistics")
print("="*70)

print(f"\nTotal job descriptions: {stats['total_jds']:,}")
print(f"Total unique skills found: {stats['unique_skills_total']:,}")

print(f"\nSkills per job description:")
print(f"  Mean: {stats['skills_per_jd']['mean']:.1f}")
print(f"  Median: {stats['skills_per_jd']['median']:.1f}")
print(f"  Min: {stats['skills_per_jd']['min']:.0f}")
print(f"  Max: {stats['skills_per_jd']['max']:.0f}")
print(f"  Std: {stats['skills_per_jd']['std']:.1f}")

print(f"\nSkills by section:")
for section, section_stats in stats['by_section'].items():
    print(f"  [{section}]")
    print(f"    Mean: {section_stats['mean']:.1f}")
    print(f"    Median: {section_stats['median']:.1f}")

print(f"\nTop 10 most common skills:")
for i, (skill, count) in enumerate(stats['top_10_skills'], 1):
    pct = (count / stats['total_jds']) * 100
    print(f"{i:2d}. {skill:40s} ({count:4d} times, {pct:.1f}%)")

## Step 7: Visualizations

In [ ]:
# Distribution of skills per JD
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_combined['num_skills'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Skills per Job Description')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Skills per JD')
axes[0].axvline(stats['skills_per_jd']['mean'], color='red', linestyle='--', label=f"Mean: {stats['skills_per_jd']['mean']:.1f}")
axes[0].legend()

# Box plot by ONET code (top 10)
top_onet = df_combined[ONET_CODE_COLUMN].value_counts().head(10).index
df_top = df_combined[df_combined[ONET_CODE_COLUMN].isin(top_onet)]
df_top.boxplot(column='num_skills', by=ONET_CODE_COLUMN, ax=axes[1], rot=45)
axes[1].set_xlabel('ONET Code')
axes[1].set_ylabel('Number of Skills')
axes[1].set_title('Skills per JD by ONET Code (Top 10)')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Skills by category
section_counts = {}
for result in results:
    for section, skills in result['by_section'].items():
        section_counts[section] = section_counts.get(section, 0) + len(skills)

plt.figure(figsize=(10, 6))
sections = list(section_counts.keys())
counts = list(section_counts.values())
plt.bar(sections, counts, edgecolor='black', alpha=0.7)
plt.xlabel('KSAO Section')
plt.ylabel('Total Skills Extracted')
plt.title('Skills Extracted by KSAO Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nSkills by section:")
for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {section:25s}: {count:6,} skills")

## Step 8: Time Series Analysis (Optional)

If you have date information, analyze skill trends over time:

In [ ]:
if 'quarter' in df_combined.columns:
    # Average skills per quarter
    quarterly_stats = df_combined.groupby('quarter')['num_skills'].agg(['mean', 'median', 'count'])
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Skills over time
    axes[0].plot(quarterly_stats.index.astype(str), quarterly_stats['mean'], marker='o', label='Mean')
    axes[0].plot(quarterly_stats.index.astype(str), quarterly_stats['median'], marker='s', label='Median')
    axes[0].set_xlabel('Quarter')
    axes[0].set_ylabel('Skills per JD')
    axes[0].set_title('Average Skills per Job Description Over Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Number of JDs per quarter
    axes[1].bar(quarterly_stats.index.astype(str), quarterly_stats['count'], edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Quarter')
    axes[1].set_ylabel('Number of Job Descriptions')
    axes[1].set_title('Sample Size per Quarter')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print("\nQuarterly statistics:")
    print(quarterly_stats)
else:
    print("No date information available for time series analysis")

## Step 9: Save Results

In [ ]:
# Save combined results
print(f"Saving results to {OUTPUT_PATH}...")
df_combined.to_parquet(OUTPUT_PATH, index=False)
print(f"✓ Saved {len(df_combined):,} records")

# Also save statistics as JSON
import json
stats_path = OUTPUT_PATH.replace('.parquet', '_stats.json')
with open(stats_path, 'w') as f:
    # Convert non-serializable objects
    stats_serializable = {
        'total_jds': stats['total_jds'],
        'unique_skills_total': stats['unique_skills_total'],
        'skills_per_jd': stats['skills_per_jd'],
        'by_section': stats['by_section'],
        'top_10_skills': [[skill, int(count)] for skill, count in stats['top_10_skills']]
    }
    json.dump(stats_serializable, f, indent=2)
print(f"✓ Saved statistics to {stats_path}")

print("\n" + "="*70)
print("EXTRACTION COMPLETE")
print("="*70)
print(f"Processed: {len(df_combined):,} job descriptions")
print(f"Extracted: {stats['unique_skills_total']:,} unique skills")
print(f"Output: {OUTPUT_PATH}")
print(f"Statistics: {stats_path}")

## Next Steps

Now that you have extracted skills, you can:

1. **Analyze specific occupations**: Filter by `onet_code` to see skill requirements for specific jobs
2. **Track skill trends**: Analyze how skill requirements change over quarters
3. **Compare occupations**: Compare skill profiles across different ONET codes
4. **Identify emerging skills**: Find skills that are increasing in frequency over time
5. **KSAO analysis**: Compare requirements across Skills, Knowledge, Abilities, and Technology categories

See `notebooks/jd_temporal_analysis.ipynb` for more advanced temporal analysis examples.